# Phase 1B - OCR content extraction (Colab runner)

Runner only: mount, git pull, install, call scripts. No logic lives here
(see CLAUDE.md P1/P2). Edit `src/` and `scripts/` locally, push, then re-run.

Pipeline: TATR grid (`tatr_predicted/`, from Phase 1A) + words -> `assign_words_to_cells`
-> filled table. Two streams, kept strictly separate (P4):
- **gt_filled** - GT structure grid + GT word tokens (`text_source=gt`); QA validation only.
- **ocr_filled** - TATR predicted grid + PaddleOCR words (`text_source=ocr`); the extraction output.

Then content metrics on spatially-aligned cells (aggregate / one-to-one / topology-matched).
**Paste back** the `processed/skipped/failed` line and the metrics JSON (text, not a screenshot).

## Boot

In [ ]:
# 1. Mount Drive (outputs persist here: gt_filled / ocr_filled / metrics / manifests).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Pull the latest code onto the VM (git is the single source of truth).
!cd /content/FinDocStructRAG 2>/dev/null && git pull || git clone https://github.com/AD2000X/FinDocStructRAG.git /content/FinDocStructRAG
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the environment.
import sys
sys.path.insert(0, '/content/FinDocStructRAG')
from src import config, fintabnet_loader
print('IN_COLAB    :', config.IN_COLAB)
print('OUTPUT_ROOT :', config.OUTPUT_ROOT)
print('cache root  :', fintabnet_loader.structure_root())

## Step 1 - gt_filled (CPU, no GPU)

Builds `gt_filled/` from the GT structure grid + GT word tokens. Pure CPU; only needs
the FinTabNet.c-Structure archive (downloaded on first run). Resumable via the manifest.
P4: gt_filled is QA validation only and is never reported as an extraction output.

In [ ]:
# Seeds are nested + issuer-unbiased (DESIGN_SPEC 18.6): debug=seed 7/limit 50,
# mvp=seed 42/limit 300, final=seed 2026/limit 500. Growing --limit reuses the smaller run.
!pip install -q huggingface_hub
!python scripts/run_phase1b_gt_filled.py --limit 300 --seed 42 --run-id mvp_rand

## Step 2 - ocr_filled (PaddleOCR)

Runs PaddleOCR over each predicted table crop, assigns word tokens to the TATR grid,
and writes `ocr_filled/` (`text_source=ocr`). Resumable: re-running skips samples already
marked success in the manifest, so a dropped session is just re-run.

On CPU PaddleOCR is the slow step (the `PP-OCRv5_server_det` detector with oneDNN disabled);
300 crops take a while. A GPU runtime is far faster - see `requirements-colab.txt` and the
GPU-paddle install note; switching the engine does not change the result (same models).

In [ ]:
# Install the OCR stack (paddleocr + engine, faiss, datasets, ...).
!pip install -q -r requirements-colab.txt

In [ ]:
# Reads tatr_predicted/<id>.json (from Phase 1A) + the crop; writes ocr_filled/<id>.json.
!python scripts/run_phase1b_ocr_filled.py --limit 300 --seed 42 --run-id mvp_rand

## Step 3 - content metrics (CPU, no GPU)

Scores ocr_filled against gt_filled on spatially-aligned cells. Three views, reported
side by side (DESIGN_SPEC 6.2): aggregate (many-to-one), one-to-one (strict), and the
topology-matched subset. `diff_content` prints per-cell GT-vs-OCR diffs for error analysis.

In [ ]:
!python scripts/evaluate_content.py --run-id mvp_rand
!python scripts/diff_content.py --run-id mvp_rand --max-samples 8

## Step 3 results - inspect artifacts

Reads the run's content report / manifests / failures. Set `run_id` to match the run
cells above. Failures are shown newest-first; `failed=0` runs add nothing here.

In [ ]:
# Inspect the latest run's artifacts (display only, no logic).
from src import config

run_id = "mvp_rand"
report    = config.EVALUATION   / f"phase1b_content_{run_id}.json"
ocr_man   = config.MANIFESTS    / f"phase1b_ocr_{run_id}.csv"
gt_man    = config.MANIFESTS    / f"phase1b_gt_{run_id}.csv"
failures  = config.FAILURE_LOGS / f"phase1b_ocr_{run_id}.jsonl"
runlog    = config.MANIFESTS    / "phase1b_ocr_runs.jsonl"

print("=== latest ocr run summary ===")
print((runlog.read_text().splitlines() or ["(empty)"])[-1] if runlog.exists() else "(none)")

print("\n=== content report ===")
print(report.read_text() if report.exists() else "(missing)")

print("\n=== ocr manifest tail (newest 5) ===")
print("\n".join(ocr_man.read_text().splitlines()[-5:]) if ocr_man.exists() else "(missing)")

print("\n=== failures (newest 5, newest first) ===")
if failures.exists():
    fl = failures.read_text().splitlines()
    print(f"total={len(fl)}")
    print("\n".join(fl[-5:][::-1]) or "(empty)")
else:
    print("(none)")

## Step 4 - spatial debug overlay (CPU, no GPU)

Overlays GT cells (green), TATR predicted cells (gray), and OCR word boxes (red) on one
crop, to see whether OCR boxes span GT column boundaries. Reads persisted tables + the
crop; runs no model. Change `--sample-id` to inspect any sample in the run.

In [ ]:
!python scripts/overlay_ocr.py --sample-id IP_2012_page_114_table_2

In [ ]:
# Display the overlay (display only, no logic).
import importlib
from src import config
importlib.reload(config)  # pick up config changes pulled after the kernel started
from IPython.display import Image as IPyImage, display

fig_root = config.FIGURES / "phase1b_overlay"
for png in sorted(fig_root.rglob("*.png")):
    print(png.relative_to(fig_root)); display(IPyImage(filename=str(png)))